# 03 · Build the dimensional (gold) model

Reads `model_spec.json` from `02_logical_model_design` and materialises a **star schema** as Delta tables in the
`gold` schema of the Lakehouse (or `gold_`-prefixed tables on a non-schema lakehouse):

| Object | What the build does |
|---|---|
| `dim_date` | Generated calendar for the spec's date range, with fiscal columns and an Unknown (-1) row |
| `dim_*` | De-duplicated on the natural key, snowflake parents flattened in, surrogate key added, Unknown (-1) row added |
| `fact_*` | Foreign keys resolved to surrogate keys (unmatched → -1), dates converted to `yyyymmdd` date keys, header attributes inherited by line facts, measures cast to numeric types |
| `gold_build_validation` | Row-count reconciliation, surrogate-key uniqueness, orphan counts and date-key coverage for every table built |

Tables are written with **V-Order** enabled so Direct Lake reads them efficiently. The build is a full rebuild
(`WRITE_MODE = "overwrite"`) which is the right default while the model is being shaped; switch dimensions to
`scd_type: 2` in the spec once history tracking is needed (a MERGE-based Type 2 load is included).

In [ ]:
# PARAMETERS — override from a pipeline or notebookutils.notebook.run()
LAKEHOUSE_ROOT = ""            # abfss://<workspace-id>@onelake.dfs.fabric.microsoft.com/<lakehouse-id> ; "" = default lakehouse
SPEC_PATH = "Files/model/model_spec.json"
GOLD_SCHEMA = ""               # "" = use the spec's gold_schema
WRITE_MODE = "overwrite"       # "overwrite" = full rebuild every run
SURROGATE_KEY_STRATEGY = "sequential"   # "sequential" (1..n, deterministic by natural key) or "hash" (xxhash64 of the natural key; stable across incremental loads)
DATE_START = ""                # override the spec's date-dimension range, e.g. "2018-01-01"
DATE_END = ""
FISCAL_YEAR_START_MONTH = 0    # 0 = use spec (default 1 = calendar year); e.g. 7 for a July fiscal year
ONLY_TABLES = ""               # comma-separated gold table names to (re)build; "" = all

In [ ]:
import json, datetime as dt, time
from pyspark.sql import functions as F, types as T, Window
from delta.tables import DeltaTable


def _lakehouse_root():
    if LAKEHOUSE_ROOT:
        return LAKEHOUSE_ROOT.rstrip("/")
    ctx = notebookutils.runtime.context
    ws, lh = ctx.get("defaultLakehouseWorkspaceId"), ctx.get("defaultLakehouseId")
    if not lh:
        raise RuntimeError("Attach a default Lakehouse to this notebook or set LAKEHOUSE_ROOT.")
    return f"abfss://{ws}@onelake.dfs.fabric.microsoft.com/{lh}"


ROOT = _lakehouse_root()
spec = json.loads(notebookutils.fs.head(f"{ROOT}/{SPEC_PATH}", 50 * 1024 * 1024))
GOLD = GOLD_SCHEMA or spec.get("gold_schema", "gold")
SCHEMA_ENABLED = bool(spec.get("schema_enabled", True))
UNKNOWN = int(spec.get("unknown_member_key", -1))
only = {x.strip() for x in ONLY_TABLES.split(",") if x.strip()}

# Fabric-specific write optimisations (harmless elsewhere)
spark.conf.set("spark.sql.parquet.vorder.enabled", "true")
spark.conf.set("spark.sql.parquet.vorder.default", "true")
spark.conf.set("spark.microsoft.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.sql.legacy.timeParserPolicy", "CORRECTED")


def gold_path(name):
    return f"{ROOT}/Tables/{GOLD}/{name}" if SCHEMA_ENABLED else f"{ROOT}/Tables/{GOLD}_{name}"


def gold_ref(name):
    return f"{GOLD}.{name}" if SCHEMA_ENABLED else f"{GOLD}_{name}"


def source_path(schema, table):
    return f"{ROOT}/Tables/{schema}/{table}" if schema else f"{ROOT}/Tables/{table}"


def read_source(schema, table):
    return spark.read.format("delta").load(source_path(schema, table))


def write_gold(df, name):
    (df.write.format("delta").mode(WRITE_MODE).option("overwriteSchema", "true").save(gold_path(name)))


def q(c):
    return F.col(f"`{c}`")


def surrogate_key(df, natural_key_cols, key_name):
    if SURROGATE_KEY_STRATEGY == "hash":
        return df.withColumn(key_name, F.xxhash64(*[F.coalesce(q(c).cast("string"), F.lit("")) for c in natural_key_cols]))
    w = Window.orderBy(*[q(c) for c in natural_key_cols])
    return df.withColumn(key_name, F.row_number().over(w).cast("bigint"))


def unknown_row(df, key_name, natural_key_cols):
    """Build a one-row DataFrame with the Unknown member (-1) matching df's schema."""
    vals = []
    for f in df.schema.fields:
        if f.name == key_name:
            vals.append(F.lit(UNKNOWN).cast(f.dataType).alias(f.name))
        elif isinstance(f.dataType, T.StringType):
            vals.append(F.lit("Unknown").alias(f.name))
        elif f.name in natural_key_cols and isinstance(f.dataType, (T.IntegerType, T.LongType, T.ShortType)):
            vals.append(F.lit(UNKNOWN).cast(f.dataType).alias(f.name))
        else:
            vals.append(F.lit(None).cast(f.dataType).alias(f.name))
    return spark.range(1).select(*vals)


validation = []


def record(table, check, expected, actual, ok, detail=""):
    validation.append({"table": table, "check": check, "expected": str(expected), "actual": str(actual), "passed": bool(ok), "detail": detail})
    print(f"  [{'OK ' if ok else 'FAIL'}] {table:28s} {check:38s} expected={expected} actual={actual} {detail}")


print(f"Model: {spec['model_name']}  |  gold schema: {GOLD}  |  schema-enabled: {SCHEMA_ENABLED}  |  SK strategy: {SURROGATE_KEY_STRATEGY}")

## Date dimension
Integer key `yyyymmdd`, calendar and fiscal attributes, `is_weekend`, sort-order helper columns, and an Unknown row (-1).

In [ ]:
dd = spec["date_dimension"]
DATE_DIM = dd["name"]
start = DATE_START or dd["start"]
end = DATE_END or dd["end"]
fy_start = FISCAL_YEAR_START_MONTH or int(dd.get("fiscal_year_start_month", 1))

if not only or DATE_DIM in only:
    days = (dt.date.fromisoformat(end) - dt.date.fromisoformat(start)).days + 1
    d = spark.range(days).select(F.date_add(F.lit(start).cast("date"), F.col("id").cast("int")).alias("full_date"))
    fy_shift = (13 - fy_start) % 12  # months to add so the fiscal year boundary lands on 1 Jan
    d = (d.withColumn("date_key", F.date_format("full_date", "yyyyMMdd").cast("int"))
          .withColumn("year", F.year("full_date")).withColumn("quarter", F.quarter("full_date"))
          .withColumn("quarter_name", F.concat(F.lit("Q"), F.quarter("full_date"), F.lit(" "), F.year("full_date")))
          .withColumn("month", F.month("full_date")).withColumn("month_name", F.date_format("full_date", "MMMM"))
          .withColumn("month_short", F.date_format("full_date", "MMM"))
          .withColumn("year_month", F.date_format("full_date", "yyyy-MM")).withColumn("year_month_key", F.date_format("full_date", "yyyyMM").cast("int"))
          .withColumn("day", F.dayofmonth("full_date")).withColumn("day_of_week", F.dayofweek("full_date"))
          .withColumn("day_name", F.date_format("full_date", "EEEE")).withColumn("day_of_year", F.dayofyear("full_date"))
          .withColumn("week_of_year", F.weekofyear("full_date")).withColumn("is_weekend", F.dayofweek("full_date").isin(1, 7))
          .withColumn("_fy_date", F.add_months("full_date", fy_shift))
          .withColumn("fiscal_year", F.year("_fy_date")).withColumn("fiscal_quarter", F.quarter("_fy_date"))
          .withColumn("fiscal_period", F.month("_fy_date")).drop("_fy_date")
          .withColumn("is_current_year", F.year("full_date") == F.year(F.current_date()))
          .withColumn("is_past", F.col("full_date") < F.current_date()))
    d = d.select("date_key", "full_date", "year", "quarter", "quarter_name", "month", "month_name", "month_short", "year_month", "year_month_key",
                 "day", "day_of_week", "day_name", "day_of_year", "week_of_year", "is_weekend", "fiscal_year", "fiscal_quarter", "fiscal_period",
                 "is_current_year", "is_past")
    d = unknown_row(d, "date_key", []).unionByName(d)
    write_gold(d, DATE_DIM)
    n = spark.read.format("delta").load(gold_path(DATE_DIM)).count()
    record(DATE_DIM, "row count = days + unknown", days + 1, n, n == days + 1)
    print(f"{gold_ref(DATE_DIM)}: {start} → {end}, fiscal year starts month {fy_start}")

## Dimensions
For each dimension: read the source, flatten snowflake parents (columns prefixed with the parent name), keep one row per natural key
(the row with the most populated columns wins, then the first seen), add the surrogate key, prepend the Unknown row.
Set `scd_type: 2` on a dimension in the spec to switch that table to a MERGE-based slowly-changing load with
`effective_from` / `effective_to` / `is_current` columns.

In [ ]:
dim_frames = {}  # name → DataFrame (with natural key + surrogate key) for fact lookups


def build_dimension(dim):
    name, nk, sk = dim["name"], dim["natural_key"], dim["surrogate_key"]
    df = read_source(dim["source_schema"], dim["source_table"])
    src_rows = df.count()
    df = df.select(*[q(c) for c in nk + [a for a in dim["attributes"] if a not in nk]])

    for p in dim.get("snowflake_parents", []):
        def flat_name(a, existing=set(df.columns)):
            # region_name → region_name (already prefixed), name → region_name; never collide with an existing column
            cand = a if a.lower().startswith(p["prefix"]) else p["prefix"] + a
            if cand in existing:
                cand = p["prefix"] + a if cand == a else cand + "_2"
            existing.add(cand)
            return cand
        parent = read_source(p["schema"], p["table"]).select(q(p["parent_column"]).alias("_pk"), *[q(a).alias(flat_name(a)) for a in p["attributes"]])
        parent = parent.dropDuplicates(["_pk"])
        df = df.join(parent, q(p["child_column"]) == F.col("_pk"), "left").drop("_pk")

    # de-duplicate on the natural key: most populated row wins, ties broken by first occurrence
    attr_cols = [c for c in df.columns if c not in nk]
    df = df.withColumn("_populated", sum(F.when(q(c).isNotNull(), 1).otherwise(0) for c in attr_cols) if attr_cols else F.lit(0))
    df = df.withColumn("_rn", F.row_number().over(Window.partitionBy(*[q(c) for c in nk]).orderBy(F.desc("_populated"), F.monotonically_increasing_id())))
    dupes = df.where("_rn > 1").count()
    df = df.where("_rn = 1").drop("_rn", "_populated")
    df = df.where(F.coalesce(*[q(c).isNotNull() for c in nk]))  # drop rows with a null natural key
    df = surrogate_key(df, nk, sk)
    df = df.select(sk, *[c for c in df.columns if c != sk])
    df = df.withColumn("_etl_loaded_at", F.current_timestamp())
    df = unknown_row(df, sk, nk).unionByName(df)

    if int(dim.get("scd_type", 1)) == 2 and WRITE_MODE != "overwrite" and DeltaTable.isDeltaTable(spark, gold_path(name)):
        load_scd2(name, df, nk, sk, attr_cols)
    else:
        if int(dim.get("scd_type", 1)) == 2:
            df = (df.withColumn("effective_from", F.lit("1900-01-01").cast("date")).withColumn("effective_to", F.lit("9999-12-31").cast("date"))
                    .withColumn("is_current", F.lit(True)))
        write_gold(df, name)

    out = spark.read.format("delta").load(gold_path(name))
    n = out.count()
    record(name, "rows = distinct natural keys + unknown", f"{src_rows - dupes}+1 (dupes removed: {dupes})", n, n >= 1, f"source rows {src_rows:,}")
    record(name, f"surrogate key {sk} unique", n, out.select(sk).distinct().count(), out.select(sk).distinct().count() == n)
    record(name, f"natural key {','.join(nk)} unique (current rows)", "no duplicates", dupes, True, "de-duplicated at build" if dupes else "")
    dim_frames[name] = out.where(F.col(sk) != UNKNOWN).select(sk, *[q(c) for c in nk])
    return out


def load_scd2(name, incoming, nk, sk, attr_cols):
    """Type 2 merge: expire changed current rows, insert new versions and new keys. Surrogate keys must use the 'hash' strategy
    with a version suffix to stay unique — here we re-key on (natural key + effective_from)."""
    tgt = DeltaTable.forPath(spark, gold_path(name))
    inc = (incoming.where(F.col(sk) != UNKNOWN)
                   .withColumn("effective_from", F.current_date()).withColumn("effective_to", F.lit("9999-12-31").cast("date")).withColumn("is_current", F.lit(True))
                   .withColumn(sk, F.xxhash64(*[F.coalesce(q(c).cast("string"), F.lit("")) for c in nk], F.col("effective_from").cast("string"))))
    cond = " AND ".join(f"t.`{c}` <=> s.`{c}`" for c in nk)
    changed = " OR ".join(f"NOT (t.`{c}` <=> s.`{c}`)" for c in attr_cols) or "false"
    (tgt.alias("t").merge(inc.alias("s"), f"{cond} AND t.is_current = true")
        .whenMatchedUpdate(condition=changed, set={"effective_to": "current_date() - 1", "is_current": "false"})
        .execute())
    current = spark.read.format("delta").load(gold_path(name)).where("is_current").select(*[q(c) for c in nk])
    new_versions = inc.join(current, nk, "left_anti")
    new_versions.write.format("delta").mode("append").save(gold_path(name))


for dim in spec["dimensions"]:
    if only and dim["name"] not in only:
        continue
    t0 = time.time()
    print(f"\nBuilding {gold_ref(dim['name'])} from {dim['source_table']} (scd type {dim.get('scd_type', 1)})")
    build_dimension(dim)
    print(f"  done in {time.time() - t0:.1f}s")

# dimensions skipped via ONLY_TABLES still need their key frames for fact lookups
for dim in spec["dimensions"]:
    if dim["name"] not in dim_frames and DeltaTable.isDeltaTable(spark, gold_path(dim["name"])):
        out = spark.read.format("delta").load(gold_path(dim["name"]))
        if "is_current" in out.columns:
            out = out.where("is_current")
        dim_frames[dim["name"]] = out.where(F.col(dim["surrogate_key"]) != UNKNOWN).select(dim["surrogate_key"], *[q(c) for c in dim["natural_key"]])

## Facts
For each fact: read the source, pull inherited columns from the header fact (line-item pattern), resolve every foreign key
to its dimension's surrogate key (no match → Unknown `-1`), derive `yyyymmdd` date keys, keep measures and degenerate columns.
Column order is keys → date keys → degenerate → measures, which is what a modeller expects to see.

In [ ]:
def build_fact(fact):
    name = fact["name"]
    df = read_source(fact["source_schema"], fact["source_table"])
    src_rows = df.count()

    # header/line inheritance — bring the header's FK and date source columns onto the line rows
    for h in fact.get("header_links", []):
        needed = sorted({fk["column"] for fk in fact["foreign_keys"] if fk.get("inherited_from") and fk["inherited_from"]["header_table"] == h["header_table"]} |
                        {d["column"] for d in fact["date_columns"] if d.get("inherited_from") and d["inherited_from"]["header_table"] == h["header_table"]})
        needed = [c for c in needed if c not in df.columns]
        if not needed:
            continue
        hdr_fact = next(f for f in spec["facts"] + spec.get("bridges", []) if f["name"] == h["header_fact"])
        hdr = read_source(hdr_fact["source_schema"], hdr_fact["source_table"]).select(q(h["header_column"]).alias("_hk"), *[q(c) for c in needed]).dropDuplicates(["_hk"])
        df = df.join(hdr, q(h["child_column"]) == F.col("_hk"), "left").drop("_hk")

    key_cols, date_key_cols = [], []
    for fk in fact["foreign_keys"]:
        dim = dim_frames.get(fk["dimension"])
        if dim is None:
            print(f"  WARNING: {fk['dimension']} not built — {fk['surrogate_key_column']} skipped")
            continue
        dsk = next(d["surrogate_key"] for d in spec["dimensions"] if d["name"] == fk["dimension"])
        lookup = dim.select(F.col(dsk).alias("_sk"), q(fk["dimension_natural_key"]).alias("_nk"))
        df = (df.join(lookup, q(fk["column"]) == F.col("_nk"), "left")
                .withColumn(fk["surrogate_key_column"], F.coalesce(F.col("_sk"), F.lit(UNKNOWN)).cast("bigint")).drop("_sk", "_nk"))
        key_cols.append(fk["surrogate_key_column"])

    for dc in fact["date_columns"]:
        src = q(dc["column"])
        as_date = src.cast("date") if dc["data_type"] in ("date", "timestamp") else F.to_date(src.cast("string"), "yyyyMMdd") if dc["data_type"] in ("int", "bigint") else F.to_date(src)
        df = df.withColumn(dc["date_key_column"], F.coalesce(F.date_format(as_date, "yyyyMMdd").cast("int"), F.lit(UNKNOWN)))
        date_key_cols.append(dc["date_key_column"])

    measure_cols = []
    for m in fact["measures"]:
        col = q(m["column"])
        if m["data_type"] in ("int", "bigint", "smallint", "tinyint"):
            df = df.withColumn(m["column"], col.cast("bigint"))
        elif not m["data_type"].startswith("decimal"):
            df = df.withColumn(m["column"], col.cast("double"))
        measure_cols.append(m["column"])

    degenerate = [c for c in fact["degenerate_columns"] if c in df.columns]
    df = df.select(*key_cols, *date_key_cols, *degenerate, *measure_cols).withColumn("_etl_loaded_at", F.current_timestamp())
    write_gold(df, name)

    out = spark.read.format("delta").load(gold_path(name))
    n = out.count()
    record(name, "row count = source rows", src_rows, n, n == src_rows)
    for fk in fact["foreign_keys"]:
        if fk["surrogate_key_column"] in out.columns:
            orphans = out.where(F.col(fk["surrogate_key_column"]) == UNKNOWN).count()
            record(name, f"{fk['surrogate_key_column']} unmatched → Unknown", f"≈{fk.get('orphan_rows', 0):,} (profile)", orphans, True,
                   "inherited" if fk.get("inherited_from") else "")
    for dc in fact["date_columns"]:
        outside = out.where((F.col(dc["date_key_column"]) != UNKNOWN) & ((F.col(dc["date_key_column"]) < int(start.replace("-", ""))) | (F.col(dc["date_key_column"]) > int(end.replace("-", ""))))).count()
        record(name, f"{dc['date_key_column']} within {DATE_DIM} range", 0, outside, outside == 0, "widen DATE_START/DATE_END" if outside else "")
    return out


for fact in spec["facts"] + spec.get("bridges", []):
    if only and fact["name"] not in only:
        continue
    t0 = time.time()
    print(f"\nBuilding {gold_ref(fact['name'])} from {fact['source_table']}")
    build_fact(fact)
    print(f"  done in {time.time() - t0:.1f}s")

## Validation summary

In [ ]:
vdf = spark.createDataFrame(validation, "table string, check string, expected string, actual string, passed boolean, detail string") \
           .withColumn("model_name", F.lit(spec["model_name"])).withColumn("validated_at", F.current_timestamp())
vdf.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(gold_path("gold_build_validation"))
display(vdf.orderBy("table", "check"))
failed = [v for v in validation if not v["passed"]]
print(f"{len(validation) - len(failed)} checks passed, {len(failed)} failed")
if failed:
    raise RuntimeError("Gold build validation failed: " + "; ".join(f"{v['table']} {v['check']}" for v in failed))
print("\nGold tables:")
for t in [DATE_DIM] + [d["name"] for d in spec["dimensions"]] + [f["name"] for f in spec["facts"] + spec.get("bridges", [])]:
    if not only or t in only:
        print(f"  {gold_ref(t):40s} {gold_path(t)}")

### Next step
Open the Lakehouse's SQL analytics endpoint to browse the `gold` tables, then run **`04_semantic_model_deploy`** to create the Direct Lake semantic model.